In [ ]:
import torch
import torch.nn as nn

class CustomGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2g = nn.Linear(input_size, 3 * hidden_size)
        self.h2g = nn.Linear(hidden_size, 3 * hidden_size)

    def forward(self, x, h_prev):
        x_gates = self.x2g(x)
        h_gates = self.h2g(h_prev)

        x_r, x_z, x_n = x_gates.chunk(3, dim=-1)
        h_r, h_z, h_n = h_gates.chunk(3, dim=-1)

        # 1. Reset Gate: r_t = \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr})
        r = torch.sigmoid(x_r + h_r)
        # 2. Update Gate: z_t = \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz})
        z = torch.sigmoid(x_z + h_z)
        # 3. New Gate: n_t = \tanh(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{t-1} + b_{hn}))
        n = torch.tanh(x_n + r * h_n)
        # 4. Hidden State: h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}
        h_next = (1 - z) * n + z * h_prev
        return h_next


class SequenceClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # Task 2 Requirement: 2-layer Bidirectional LSTM backbone with dropout=0.5
        self.lstm = nn.LSTM(input_size=input_size,hidden_size=hidden_size,num_layers=2,batch_first=True,bidirectional=True,dropout=0.5)
        # Task 2 Requirement: Linear head mapping concatenated final hidden states
        # (2 * hidden_size due to bidirectional nature) to num_classes
        self.fc = nn.Linear(2 * hidden_size, num_classes)
    def forward(self, x):
        """Standard forward pass for uniform-length sequences[cite: 1]."""
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Extract and concatenate top-layer final forward and backward hidden states
        # h_n shape: (num_layers * num_directions, batch_size, hidden_size)
        h_forward = h_n[-2, :, :]
        h_backward = h_n[-1, :, :]
        h_cat = torch.cat((h_forward, h_backward), dim=1)
        return self.fc(h_cat)
    def forward_efficient(self, x_padded, lengths):
        """Task 3: Efficient forward pass for variable-length padded sequences[cite: 1]."""
        # Pack padded sequence to ignore computation on padding tokens
        packed_x = nn.utils.rnn.pack_padded_sequence(x_padded,lengths.cpu(),batch_first=True, enforce_sorted=False)
        packed_out, (h_n, c_n) = self.lstm(packed_x)
        h_forward = h_n[-2, :, :]
        h_backward = h_n[-1, :, :]
        h_cat = torch.cat((h_forward, h_backward), dim=1)
        return self.fc(h_cat)

import numpy as np
import pandas as pd
import yfinance as yf
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense # Changed LSTM to GRU
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f"TensorFlow Version: {tf.__version__}")

ticker = "AAPL"
print(f"Downloading historical data for {ticker}...")

df = yf.download(ticker, start="2019-01-01", end="2024-01-01", auto_adjust=False)

data = df[['Close']].values

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

def create_sequences(dataset, lookback=10):
    X, y = [], []
    for i in range(lookback, len(dataset)):
        X.append(dataset[i - lookback : i, 0])
        y.append(dataset[i, 0])
    return np.array(X), np.array(y)

LOOKBACK = 10
X, y = create_sequences(scaled_data, lookback=LOOKBACK)

X = np.reshape(X, (X.shape[0], X.shape[1], 1))
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

model_gru = Sequential([
    GRU(32, return_sequences=False, input_shape=(LOOKBACK, 1)), # Changed LSTM to GRU
    Dense(16, activation='relu'),
    Dense(1)
])

model_gru.summary()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model_gru.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])
epochs = 30
batch_size = 16

print("\nTraining GRU on real-world stock data...") # Updated print statement
history = model_gru.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test),
    verbose=1
)

predictions_scaled = model_gru.predict(X_test)
predictions = scaler.inverse_transform(predictions_scaled)
actuals = scaler.inverse_transform(y_test.reshape(-1, 1))

mse = mean_squared_error(actuals, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actuals, predictions)
mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
r2 = r2_score(actuals, predictions)

print("\n" + "="*40)
print("       EVALUATION METRICS (TEST SET)       ")
print("="*40)
print(f"Root Mean Squared Error (RMSE) : ${rmse:.2f}")
print(f"Mean Absolute Error (MAE)      : ${mae:.2f}")
print(f"Mean Absolute Percentage Error : {mape:.2f}%")
print(f"R² Score (Variance Explained)   : {r2:.4f}")
print("="*40)

print("\nFirst 5 Sample Predictions (Predicted vs Actual AAPL Close Price):")
for i in range(5):
    print(f"Day {i+1}: Predicted = ${predictions[i][0]:.2f} | Actual = ${actuals[i][0]:.2f}")
if __name__ == "__main__":
    print("--- Running Verification Checks ---")

    # 1. Custom GRU Cell Check
    gru_cell = CustomGRUCell(input_size=16, hidden_size=32)
    x_step = torch.randn(8, 16)       # (Batch, Feat)
    h_prev = torch.zeros(8, 32)       # (Batch, Hidden)
    h_next = gru_cell(x_step, h_prev)
    print(f"Task 1 - CustomGRUCell output shape: {h_next.shape}")  # Expected: (8, 32)

    # 2. SequenceClassifier & Efficient Forward Check
    classifier = SequenceClassifier(input_size=16, hidden_size=32, num_classes=5)
    x_padded = torch.randn(8, 10, 16)  # (Batch, Seq_Len, Feat)
    lengths = torch.tensor([10, 9, 8, 7, 6, 5, 4, 3])

    out_standard = classifier(x_padded)
    out_efficient = classifier.forward_efficient(x_padded, lengths)

    print(f"Task 2 - Standard Forward Output shape: {out_standard.shape}")    # Expected: (8, 5)
    print(f"Task 3 - Efficient Forward Output shape: {out_efficient.shape}")  # Expected: (8, 5)
    print("\nAll checks completed successfully!")

--- Running Verification Checks ---
Task 1 - CustomGRUCell output shape: torch.Size([8, 32])
Task 2 - Standard Forward Output shape: torch.Size([8, 5])
Task 3 - Efficient Forward Output shape: torch.Size([8, 5])

All checks completed successfully!
